# DetaNet ELoRA + AdaLoRA Smoke Train
This notebook loads a single shard, builds DetaNet with ELoRA (equivariant LoRA) + AdaLoRA (scalar/attention), and runs a forward/backward loop for 10 epochs.

- Adjust the config block to scale up.
- Requires: `torch`, `torch_geometric`, `e3nn` (via vendored ELoRA), and `peft` for AdaLoRA.


**Note:** If you start from random weights, keep `ADAPTER_FREEZE_BASE=False` or load a pretrained checkpoint before freezing.

In [1]:
import os
import sys
from pathlib import Path

import torch
from torch_geometric.loader import DataLoader

# Make repo modules importable
REPO_ROOT = Path.cwd().parent
MODEL_ROOT = REPO_ROOT / 'capsule-3259363' / 'code'
sys.path.insert(0, str(MODEL_ROOT))

from detanet_model.detanet import DetaNet

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cpu')

In [2]:
# Config (keep small for a quick smoke test)
NUM_FEATURES = 128
MAXL = 3
NUM_BLOCK = 3
NUM_RADIAL = 32
ATTN_HEAD = 8
BATCH_SIZE = 2
EPOCHS = 10

ELORA_PATH = 'vendored'  # uses third_party/ELoRA
USE_ADALORA = True

EXCLUDE_KEYS = [
    'field_imputed', 'field_generated', 'field_confidence', 'field_source',
    'smile', 'source'
]

SHARD_PATH = REPO_ROOT / 'samples' / 'shard_000013.pt'
SHARD_PATH
ADAPTER_FREEZE_BASE = True
CHECKPOINT_PATH = MODEL_ROOT / 'trained_param/latest/latest_Hij.pth'

In [3]:
MASK_MODE = globals().get('MASK_MODE', 'binary')
USE_IMPUTE_MASK = globals().get('USE_IMPUTE_MASK', True)
NORMALIZE = 'dataset_per_atom'
IMPUTED_WEIGHT = 0.2  # fallback weight for imputed targets when confidence missing


In [4]:
# AdaLoRA config (if peft is installed)
adalora_cfg = None
if USE_ADALORA:
    try:
        from peft import AdaLoraConfig, TaskType
        adalora_cfg = AdaLoraConfig(
            r=4,
            lora_alpha=16,
            tinit=10,
            tfinal=20,
            total_step=100,
            lora_dropout=0.05,
            task_type=TaskType.FEATURE_EXTRACTION,
        )
    except Exception as exc:
        print('peft not available, skipping AdaLoRA:', exc)
        adalora_cfg = None

model = DetaNet(
    num_features=NUM_FEATURES,
    maxl=MAXL,
    num_block=NUM_BLOCK,
    num_radial=NUM_RADIAL,
    attention_head=ATTN_HEAD,
    scalar_outsize=1,
    irreps_out=None,
    out_type='scalar',
    summation=True,
    elora_path=ELORA_PATH,
    adalora_config=adalora_cfg,
    adapter_freeze_base=ADAPTER_FREEZE_BASE,
    device=device,
).to(device)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f'Trainable params: {trainable} / {total}')
# Load checkpoint if provided (non-strict to allow LoRA params)
if CHECKPOINT_PATH is not None:
    ckpt = torch.load(CHECKPOINT_PATH, map_location='cpu', weights_only=False)
    if isinstance(ckpt, dict):
        if 'state_dict' in ckpt and isinstance(ckpt['state_dict'], dict):
            ckpt = ckpt['state_dict']
        elif 'model' in ckpt and isinstance(ckpt['model'], dict):
            ckpt = ckpt['model']
        elif 'module' in ckpt and isinstance(ckpt['module'], dict):
            ckpt = ckpt['module']
    if isinstance(ckpt, dict) and all(k.startswith('module.') for k in ckpt):
        ckpt = {k[len('module.'):] : v for k, v in ckpt.items()}
    missing, unexpected = model.load_state_dict(ckpt, strict=False)
    print(f'loaded checkpoint {CHECKPOINT_PATH}')
    print('missing keys', len(missing), 'unexpected keys', len(unexpected))

# Sanity: ensure config matches checkpoint shapes
if CHECKPOINT_PATH is not None:
    if isinstance(ckpt, dict) and 'Embedding.nuclare_emb.weight' in ckpt:
        inferred = ckpt['Embedding.nuclare_emb.weight'].shape[1]
        if inferred != NUM_FEATURES:
            raise ValueError(f'NUM_FEATURES={NUM_FEATURES} does not match checkpoint ({inferred}).')


/Users/rahul/Desktop/hp-proteins-ml/.venv/lib/python3.13/site-packages/torch/jit/_check.py:178: UserWarning: The TorchScript type system doesn't support instance-level annotations on empty non-base types in `__init__`. Instead, either 1) use a type annotation in the class body, or 2) wrap the type in `torch.jit.Attribute`.
  warnings.warn(
/Users/rahul/Desktop/hp-proteins-ml/.venv/lib/python3.13/site-packages/torch/jit/_check.py:178: UserWarning: The TorchScript type system doesn't support instance-level annotations on empty non-base types in `__init__`. Instead, either 1) use a type annotation in the class body, or 2) wrap the type in `torch.jit.Attribute`.
  warnings.warn(
/Users/rahul/Desktop/hp-proteins-ml/.venv/lib/python3.13/site-packages/torch/jit/_check.py:178: UserWarning: The TorchScript type system doesn't support instance-level annotations on empty non-base types in `__init__`. Instead, either 1) use a type annotation in the class body, or 2) wrap the type in `torch.jit.Att

Trainable params: 240084 / 1646243
loaded checkpoint /Users/rahul/Desktop/hp-proteins-ml/capsule-3259363/code/trained_param/latest/latest_Hij.pth
missing keys 192 unexpected keys 28


/Users/rahul/Desktop/hp-proteins-ml/.venv/lib/python3.13/site-packages/torch/jit/_check.py:178: UserWarning: The TorchScript type system doesn't support instance-level annotations on empty non-base types in `__init__`. Instead, either 1) use a type annotation in the class body, or 2) wrap the type in `torch.jit.Attribute`.
  warnings.warn(
/Users/rahul/Desktop/hp-proteins-ml/.venv/lib/python3.13/site-packages/torch/jit/_check.py:178: UserWarning: The TorchScript type system doesn't support instance-level annotations on empty non-base types in `__init__`. Instead, either 1) use a type annotation in the class body, or 2) wrap the type in `torch.jit.Attribute`.
  warnings.warn(
/Users/rahul/Desktop/hp-proteins-ml/.venv/lib/python3.13/site-packages/torch/jit/_check.py:178: UserWarning: The TorchScript type system doesn't support instance-level annotations on empty non-base types in `__init__`. Instead, either 1) use a type annotation in the class body, or 2) wrap the type in `torch.jit.Att

In [5]:
# Load shard and build DataLoader
MASK_MODE = globals().get('MASK_MODE', 'confidence')
USE_IMPUTE_MASK = globals().get('USE_IMPUTE_MASK', True)
IMPUTED_WEIGHT = globals().get('IMPUTED_WEIGHT', 0.2)
data_list = torch.load(SHARD_PATH, map_location='cpu', weights_only=False)


def attach_mask(item, task='energy'):
    if not USE_IMPUTE_MASK:
        target = getattr(item, task)
        item.__setattr__(f'mask_{task}', torch.ones_like(target))
        return
    mask_value = 1.0
    imputed = getattr(item, 'field_imputed', None)
    if isinstance(imputed, dict) and imputed.get(task, False):
        if MASK_MODE == 'binary':
            mask_value = IMPUTED_WEIGHT
        else:
            mask_value = None
    if MASK_MODE == 'confidence':
        conf = getattr(item, 'field_confidence', None)
        if isinstance(conf, dict) and conf.get(task, None) is not None:
            mask_value = float(conf.get(task))
        elif mask_value is None:
            mask_value = IMPUTED_WEIGHT
    target = getattr(item, task)
    item.__setattr__(f'mask_{task}', torch.full_like(target, float(mask_value)))

for item in data_list:
    attach_mask(item, task='energy')

loader = DataLoader(
    data_list,
    batch_size=BATCH_SIZE,
    shuffle=True,
    exclude_keys=EXCLUDE_KEYS,
)
len(data_list)


50

In [6]:
# Optimizer (AdamW default; replace if you want DistributedShampoo)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.0)

def compute_stats(loader):
    total = 0.0
    total_sq = 0.0
    count = 0.0
    for batch in loader:
        batch = batch.to(device)
        target = batch.energy.float()
        mask = getattr(batch, 'mask_energy', None)
        if mask is None or not USE_IMPUTE_MASK:
            mask = torch.ones_like(target)
        else:
            mask = mask.float()
        if mask.sum() == 0:
            mask = torch.ones_like(target)
        if mask.sum() == 0:
            mask = torch.ones_like(target)
        if 'per_atom' in NORMALIZE:
            counts = torch.bincount(batch.batch, minlength=target.shape[0]).float().to(device)
            while counts.dim() < target.dim():
                counts = counts.unsqueeze(-1)
            target = target / counts.clamp(min=1.0)
        total += (target * mask).sum().item()
        total_sq += (target * target * mask).sum().item()
        count += mask.sum().item()
    if count == 0:
        return torch.tensor(0.0, device=device), torch.tensor(1.0, device=device)
    mean = total / count
    var = max(total_sq / count - mean ** 2, 0.0)
    std = (var ** 0.5) if var > 0 else 1.0
    return torch.tensor(mean, device=device), torch.tensor(std, device=device)

norm_mean = torch.tensor(0.0, device=device)
norm_std = torch.tensor(1.0, device=device)
if NORMALIZE.startswith('dataset'):
    norm_mean, norm_std = compute_stats(loader)

for epoch in range(EPOCHS):
    model.train()
    running = 0.0
    for batch in loader:
        batch = batch.to(device)
        target = batch.energy.float()
        mask = getattr(batch, 'mask_energy', None)
        if mask is None or not USE_IMPUTE_MASK:
            mask = torch.ones_like(target)
        else:
            mask = mask.float()
        if mask.sum() == 0:
            mask = torch.ones_like(target)
        if mask.sum() == 0:
            mask = torch.ones_like(target)
        optimizer.zero_grad(set_to_none=True)
        pred = model(z=batch.z, pos=batch.pos, edge_index=batch.edge_index, batch=batch.batch).float()

        if 'per_atom' in NORMALIZE:
            counts = torch.bincount(batch.batch, minlength=target.shape[0]).float().to(device)
            while counts.dim() < target.dim():
                counts = counts.unsqueeze(-1)
            pred = pred / counts.clamp(min=1.0)
            target = target / counts.clamp(min=1.0)

        if NORMALIZE.startswith('batch'):
            denom = mask.sum().clamp(min=1.0)
            mean = (target * mask).sum() / denom
            var = ((target - mean) ** 2 * mask).sum() / denom
            std = torch.sqrt(var + 1e-12)
        elif NORMALIZE.startswith('dataset'):
            mean = norm_mean
            std = norm_std
        else:
            mean = 0.0
            std = 1.0

        pred = (pred - mean) / std
        target = (target - mean) / std

        loss = ((pred - target) ** 2 * mask).sum() / mask.sum().clamp(min=1.0)
        loss.backward()
        optimizer.step()
        running += loss.item()
    print(f'epoch={epoch} loss={running / max(1, len(loader)):.6f}')

epoch=0 loss=57.273076
epoch=1 loss=57.264211
epoch=2 loss=57.220667
epoch=3 loss=57.091350
epoch=4 loss=56.783428
epoch=5 loss=55.940497
epoch=6 loss=53.873302
epoch=7 loss=44.358747
epoch=8 loss=7.840614
epoch=9 loss=3.084464
